In [ ]:
!pip install wandb -q
import wandb
wandb.login()

In [ ]:
import os
import zipfile
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import collections
from torchvision import transforms
import matplotlib.pyplot as plt
from PIL import Image
from collections import Counter
import collections
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, WeightedRandomSampler
import collections
from torchvision import models
import numpy as np
from sklearn.metrics import average_precision_score
from tqdm import tqdm
import shutil
import random
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
    confusion_matrix
)
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR
from PIL import Image
from torchvision import transforms





In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
zip_path = "/content/drive/MyDrive/disease/train.zip.zip"
val_zip = "/content/drive/MyDrive/disease/val.zip.zip"
extract_path = "/content/data"

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)
with zipfile.ZipFile(val_zip, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Unzipped successfully!")


In [ ]:
TRAIN_PATH = "/content/data/train"
VAL_PATH   = "/content/data/val"
SAVE_PATH = "/content/drive/MyDrive/disease/best_model2.pth"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

In [ ]:
label_map = {
    'black rot': 0, 'mosaic virus': 1, 'rust': 2, 'scab': 3,
    'anthracnose': 4, 'black leaf streak': 5, 'bunchy top': 6,
    'downy mildew': 7, 'pepper blossom end rot': 8,
    'pepper powdery mildew': 9, 'alternaria leaf spot': 10,
    'early blight': 11, 'leaf spot': 12, 'powdery mildew': 13,
    'canker': 14, 'greening disease': 15, 'berry blotch': 16,
    'leaf rust': 17, 'gray leaf spot': 18, 'northern leaf blight': 19,
    'smut': 20, 'angular leaf spot': 21, 'bacterial wilt': 22,
    'sheath blight': 23, 'tar spot': 24, 'brown rot': 25,
    'leaf curl': 26, 'late blight': 27, 'brown spot': 28,
    'frog eye leaf spot': 29, 'mosaic': 30, 'bacterial leaf spot': 31,
    'leaf mold': 32, 'septoria leaf spot': 33,
    'bacterial leaf streak (black chaff)': 34, 'head scab': 35,
    'loose smut': 36, 'septoria blotch': 37, 'stem rust': 38,
    'stripe rust': 39,
}
label_to_name = {v: k for k, v in label_map.items()}
num_classes   = len(label_map)
print(f" label_map : {num_classes} classes")

In [ ]:
class PlantDataset(Dataset):
    def __init__(self, root_dir, label_map, transform=None):
        self.root_dir = root_dir
        self.label_map = label_map
        self.transform = transform
        self.image_paths = []
        self.labels = []
        self.classes = []

        for c in sorted(os.listdir(root_dir)):
            if c.startswith(".") or not os.path.isdir(os.path.join(root_dir, c)):
                continue

            disease = " ".join(c.split(" ")[1:])
            if disease not in label_map:
                continue

            label = label_map[disease]
            if disease not in self.classes:
                self.classes.append(disease)

            class_path = os.path.join(root_dir, c)
            for img in os.listdir(class_path):
                if img.lower().endswith((".jpg", ".jpeg", ".png")):
                    self.image_paths.append(os.path.join(class_path, img))
                    self.labels.append(label)

        print(f"{root_dir} → {len(self.image_paths)} images | {len(self.classes)} classes")

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx: int):
        if len(self.image_paths) == 0:
            raise RuntimeError("Dataset is empty — check your root_dir and label_map")

        img_path = self.image_paths[idx % len(self.image_paths)]

        try:
            img = Image.open(img_path).convert("RGB")
        except Exception as e:
            print(f"  [WARN] Corrupt image skipped: {img_path} ({e})")
            return self.__getitem__((idx + 1) % len(self.image_paths))

        label = self.labels[idx % len(self.image_paths)]

        if self.transform:
            img = self.transform(img)

        return img, label

    def get_sample_weights(self, sqrt_balance=True):
        counts = Counter(self.labels)
        weights = [1.0 / (counts[l] ** 0.5 if sqrt_balance else counts[l]) for l in self.labels]
        mean_w = sum(weights) / len(weights)
        return [w / mean_w for w in weights]

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomResizedCrop(224, scale=(0.85, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.ColorJitter(
        brightness=0.15,
        contrast=0.15,
        saturation=0.1,
        hue=0.01
    ),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])

In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0):
        super().__init__()
        self.gamma = gamma

    def forward(self, inputs, targets):
        ce = F.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-ce)
        loss = (1 - pt) ** self.gamma * ce
        return loss.mean()

criterion = FocalLoss(gamma=2.0)
print("FocalLoss ready")

In [ ]:
train_dataset = PlantDataset(TRAIN_PATH, label_map, train_transform)
val_dataset = PlantDataset(VAL_PATH, label_map, val_transform)

train_classes = set(train_dataset.labels)
val_classes = set(val_dataset.labels)
missing = train_classes - val_classes

if missing:
    label_to_name = {v: k for k, v in label_map.items()}

    def find_class_folder(root, disease_name):
        for folder in sorted(os.listdir(root)):
            suffix = " ".join(folder.split(" ")[1:])
            if suffix == disease_name:
                return os.path.join(root, folder)
        return None

    for label_idx in missing:
        disease_name = label_to_name.get(label_idx)
        if not disease_name:
            print(f"  [WARN] No name found for label {label_idx}, skipping")
            continue

        print(f" Missing in val: class {label_idx} → '{disease_name}'")

        train_class_path = find_class_folder(TRAIN_PATH, disease_name)
        if not train_class_path:
            print(f"  [ERROR] Folder not found in train for '{disease_name}'")
            continue

        folder_name = os.path.basename(train_class_path)
        val_class_path = os.path.join(VAL_PATH, folder_name)
        os.makedirs(val_class_path, exist_ok=True)

        images = [
            f for f in sorted(os.listdir(train_class_path))
            if f.lower().endswith((".jpg"))
        ]

        num_to_move = max(5, min(10, int(len(images) * 0.2)))

        images_to_move = random.sample(images, num_to_move)

        for img in images_to_move:
            shutil.move(
                os.path.join(train_class_path, img),
                os.path.join(val_class_path, img)
            )

        print(f" Moved {num_to_move}/{len(images)} images → {val_class_path}")

    train_dataset = PlantDataset(TRAIN_PATH, label_map, train_transform)
    val_dataset   = PlantDataset(VAL_PATH,   label_map, val_transform)

    still_missing = set(train_dataset.labels) - set(val_dataset.labels)
    if still_missing:
        print(f"  [WARN] Still missing after fix: {still_missing}")
    else:
        print(" All classes now present in both splits")

else:
    print("All train classes present in val — no fix needed")

print(" Train class distribution:")
for label, count in sorted(Counter(train_dataset.labels).items(), key=lambda x: x[1], reverse=True):
    print(f"  Class {label}: {count} images")

values = list(Counter(train_dataset.labels).values())
print(f"\n  Min: {min(values)} | Max: {max(values)} | Mean: {sum(values)/len(values):.1f}")



In [ ]:
values = list(Counter(train_dataset.labels).values())
print(f"Min: {min(values)} | Max: {max(values)} | Imbalance: {max(values)/min(values):.1f}:1")

sample_weights = train_dataset.get_sample_weights(sqrt_balance=True)
sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

pin = torch.cuda.is_available()
train_loader = DataLoader(train_dataset, batch_size=64, sampler=sampler,
                          num_workers=2, pin_memory=pin)
val_loader   = DataLoader(val_dataset, batch_size=64, shuffle=False,
                          num_workers=2, pin_memory=pin)
print("DataLoaders ready")

In [ ]:
counter = collections.Counter(train_dataset.labels)
print("Most common:", counter.most_common(5))
print("Least common:", counter.most_common()[-5:])

values = list(counter.values())
print(f"Imbalance ratio: {max(values)/min(values):.1f}:1")

In [ ]:
model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)

in_features = model.fc.in_features

model.fc = nn.Sequential(
    nn.Dropout(p=0.4),
    nn.Linear(in_features, 512),
    nn.ReLU(),
    nn.Dropout(p=0.3),
    nn.Linear(512, num_classes)
)

model = model.to(device)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())

print(f"Trainable: {trainable:,} / {total:,} params ({100*trainable/total:.1f}%)")
print(f"Output classes: {num_classes}")

wandb.init(
    project="plant-disease-classification",
    name="resnet50_focal_sqrt_cosine",
    config={
        "model": "resnet50",
        "batch_size": 64,
        "frozen_epochs": 10,
        "fine_tune_epochs": 30,
        "frozen_lr": 1e-3,
        "fine_tune_lr": 1e-4,
        "weight_decay": 1e-4,
        "scheduler": "CosineAnnealingLR",
        "loss": "FocalLoss(gamma=2.0)",
        "sampler": "WeightedRandomSampler(sqrt)",
        "dropout": 0.4,
        "augmentation": "RandomResizedCrop(0.85)+ColorJitter",
        "num_classes": num_classes
    }
)


In [ ]:
def mixup_data(x, y, alpha=0.2):
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1

    batch_size = x.size()[0]
    index = torch.randperm(batch_size).to(x.device)

    mixed_x = lam * x + (1 - lam) * x[index, :]
    y_a, y_b = y, y[index]
    return mixed_x, y_a, y_b, lam

def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, device, mixup_alpha=0.4):
    model.train()
    total_loss = 0.0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        images, labels_a, labels_b, lam = mixup_data(images, labels, alpha=mixup_alpha)

        optimizer.zero_grad()

        outputs = model(images)

        loss = mixup_criterion(criterion, outputs, labels_a, labels_b, lam)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

In [ ]:
def validate_detailed(model, loader, criterion, device, num_classes, return_confusion=False, label_names=None):
    model.eval()

    total_loss = 0
    all_preds = []
    all_labels = []
    all_probs = []

    with torch.no_grad():
        for images, labels in tqdm(loader, desc="Validating", leave=False):
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            outputs = model(images)
            loss = criterion(outputs, labels)
            total_loss += loss.item()

            probs = torch.softmax(outputs, dim=1)
            preds = torch.argmax(probs, dim=1)

            all_preds.append(preds.cpu().numpy())
            all_labels.append(labels.cpu().numpy())
            all_probs.append(probs.cpu().numpy())

    all_preds = np.concatenate(all_preds)
    all_labels = np.concatenate(all_labels)
    all_probs = np.concatenate(all_probs)

    accuracy = (all_preds == all_labels).mean()

    precision_macro = precision_score(all_labels, all_preds, average='macro', zero_division=0)
    recall_macro = recall_score(all_labels, all_preds, average='macro', zero_division=0)
    f1_macro = f1_score(all_labels, all_preds, average='macro', zero_division=0)

    precision_weighted = precision_score(all_labels, all_preds, average='weighted', zero_division=0)
    recall_weighted = recall_score(all_labels, all_preds, average='weighted', zero_division=0)
    f1_weighted = f1_score(all_labels, all_preds, average='weighted', zero_division=0)

    labels_onehot = np.zeros((len(all_labels), num_classes), dtype=np.int32)
    labels_onehot[np.arange(len(all_labels)), all_labels] = 1

    per_class_ap = {}
    for i in range(num_classes):
        if labels_onehot[:, i].sum() > 0:
            per_class_ap[i] = average_precision_score(labels_onehot[:, i], all_probs[:, i])
        else:
            per_class_ap[i] = float('nan')

    valid_aps = [v for v in per_class_ap.values() if not np.isnan(v)]
    mAP = float(np.mean(valid_aps)) if valid_aps else 0.0


    valid_classes = [(k, v) for k, v in per_class_ap.items() if not np.isnan(v)]
    if valid_classes:
        worst_class, worst_ap = min(valid_classes, key=lambda x: x[1])
        best_class, best_ap = max(valid_classes, key=lambda x: x[1])

    results = {
          'loss': total_loss / len(loader),
          'accuracy': accuracy,
          'mAP': mAP,
          'f1_macro': f1_macro,
          'precision_macro': precision_macro,
          'recall_macro': recall_macro,
          'precision_weighted': precision_weighted,
          'recall_weighted': recall_weighted,
          'f1_weighted': f1_weighted,
          'per_class_ap': per_class_ap,
          'predictions': all_preds,
          'labels': all_labels,
      }

    if return_confusion:
        results['confusion_matrix'] = confusion_matrix(all_labels, all_preds)

    model.train()
    return results

In [ ]:

def compute_map(model, loader, device, num_classes, return_per_class=False):

    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in tqdm(loader, desc="Computing mAP", leave=False):
            images = images.to(device)
            outputs = model(images)
            probs = torch.softmax(outputs, dim=1)

            all_preds.append(probs.cpu().numpy())
            all_labels.append(labels.cpu().numpy())

    preds = np.vstack(all_preds)
    labels = np.hstack(all_labels)

    labels_onehot = np.zeros((len(labels), num_classes), dtype=np.int32)
    labels_onehot[np.arange(len(labels)), labels] = 1

    aps = {}
    for i in range(num_classes):
        if labels_onehot[:, i].sum() > 0:
            aps[i] = average_precision_score(labels_onehot[:, i], preds[:, i])
        else:
            aps[i] = float('nan')

    valid_aps = [v for v in aps.values() if not np.isnan(v)]
    mean_ap = float(np.mean(valid_aps)) if valid_aps else 0.0

    print(f" mAP: {mean_ap:.4f}")

    sorted_aps = sorted([(k, v) for k, v in aps.items() if not np.isnan(v)], key=lambda x: x[1])
    print(f"Worst class (AP {sorted_aps[0][1]:.3f}): {sorted_aps[0][0]}")
    print(f"Best class (AP {sorted_aps[-1][1]:.3f}): {sorted_aps[-1][0]}")

    model.train()

    if return_per_class:
        return mean_ap, aps
    return mean_ap

In [ ]:
for param in model.parameters():
    param.requires_grad = False

for param in model.fc.parameters():
    param.requires_grad = True

optimizer = optim.Adam(
    model.fc.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)

scheduler = CosineAnnealingLR(
    optimizer,
    T_max=10
)

epochs = 10

for epoch in range(epochs):

    train_loss = train_one_epoch(
        model, train_loader, optimizer, criterion, device
    )

    metrics = validate_detailed(
        model, val_loader, criterion, device, num_classes
    )

    scheduler.step()

    wandb.log({
        "stage": "frozen",
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "val_loss": metrics['loss'],
        "val_accuracy": metrics['accuracy'],
        "val_mAP": metrics['mAP'],
        "val_f1": metrics['f1_macro'],
        "learning_rate": optimizer.param_groups[0]['lr'],
    })

    print(f"\n{'─'*50}")
    print(f"[Frozen] Epoch {epoch + 1}/{epochs}")
    print(f"  LR:          {optimizer.param_groups[0]['lr']:.2e}")
    print(f"  Train Loss:  {train_loss:.4f}")
    print(f"  Val Loss:    {metrics['loss']:.4f}")
    print(f"  mAP:         {metrics['mAP']:.4f}   ")
    print(f"  F1 Macro:    {metrics['f1_macro']:.4f}   ")
    print(f"  F1 Weighted: {metrics['f1_weighted']:.4f}   ")
    print(f"{'─'*50}")

In [ ]:
for param in model.parameters():
    param.requires_grad = True

EPOCHS = 30
FINE_TUNE_LR = 1e-4
EARLY_STOP = 7
MIXUP_ALPHA = 0.2

optimizer = optim.Adam(
    model.parameters(),
    lr=FINE_TUNE_LR,
    weight_decay=1e-4
)

scheduler = CosineAnnealingLR(
    optimizer,
    T_max = EPOCHS,
    eta_min = 1e-6
)

best_map = 0.0
best_metrics = {}
epochs_no_improve = 0

for epoch in range(EPOCHS):

    train_loss = train_one_epoch(
        model, train_loader, optimizer, criterion, device,
        mixup_alpha=MIXUP_ALPHA
    )


    metrics = validate_detailed(
        model, val_loader, criterion, device,
        num_classes=num_classes,
        return_confusion=True,
        label_names={v: k for k, v in label_map.items()}
    )

    scheduler.step()

    cm = metrics['confusion_matrix']
    per_class_acc = cm.diagonal() / (cm.sum(axis=1) + 1e-8)


    print(f"\n{'─'*55}")
    print(f"[Fine-tune] Epoch {epoch+1}/{EPOCHS}")
    print(f"  LR:            {optimizer.param_groups[0]['lr']:.2e}")
    print(f"  Train Loss:    {train_loss:.4f}")
    print(f"  Val Loss:      {metrics['loss']:.4f}")
    print(f"  mAP:           {metrics['mAP']:.4f}   ")
    print(f"  F1 Macro:      {metrics['f1_macro']:.4f}  ")
    print(f"  F1 Weighted:   {metrics['f1_weighted']:.4f} ")
    print(f"  Precision:     {metrics['precision_macro']:.4f}")
    print(f"  Recall:        {metrics['recall_macro']:.4f}")
    print(f"  Mean cls acc:  {per_class_acc.mean():.4f}")
    print(f"  Worst cls acc: {per_class_acc.min():.4f} "
          f"(class {per_class_acc.argmin()})")

    if 'per_class_ap' in metrics:
        worst = sorted(metrics['per_class_ap'].items(), key=lambda x: x[1])[:3]
        print(f"  Worst 3 AP:    {[(c, f'{ap:.2f}') for c, ap in worst]}")

    print(f"{'─'*55}")

    wandb.log({
        "stage":           "finetune",
        "epoch":           epoch + 1,
        "learning_rate":   optimizer.param_groups[0]['lr'],
        "train_loss":      train_loss,
        "val_loss":        metrics['loss'],
        "val_mAP":         metrics['mAP'],
        "val_f1_macro":    metrics['f1_macro'],
        "val_f1_weighted": metrics['f1_weighted'],
        "val_precision":   metrics['precision_macro'],
        "val_recall":      metrics['recall_macro'],
        "mean_class_acc":  per_class_acc.mean(),
        "worst_class_acc": per_class_acc.min(),
    })


    if metrics['mAP'] > best_map:
        best_map     = metrics['mAP']
        best_metrics = {
            'mAP':       metrics['mAP'],
            'f1_macro':  metrics['f1_macro'],
            'precision': metrics['precision_macro'],
            'recall':    metrics['recall_macro'],
        }
        torch.save({
            'model_state_dict': model.state_dict(),
            'label_map':        label_map,
            'num_classes':      num_classes,
            'architecture':     'resnet50',
        }, SAVE_PATH)

        print(f"  Saved best model (mAP: {best_map:.4f}) → {SAVE_PATH}")
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1
        print(f"  No improvement ({epochs_no_improve}/{EARLY_STOP})")
        if epochs_no_improve >= EARLY_STOP:
            print(" Early stopping triggered")
            break

print(" BEST RESULTS (Fine-tune Stage):")

for k, v in best_metrics.items():
    print(f"  {k:<12}: {v:.4f}")



In [ ]:
cm = metrics['confusion_matrix']
classes = [33, 31, 2]
for i in classes:
    print(f"Class {i} predictions: {cm[i][classes]}")

In [ ]:
print(label_to_name[31])
print(label_to_name[33])
print(label_to_name[2])

In [ ]:
def predict(image_path, model, label_map, device, threshold=0.4):

    label_to_name = {v: k for k, v in label_map.items()}

    transform = transforms.Compose([
      transforms.Resize((256, 256)),
      transforms.CenterCrop(224),
      transforms.ToTensor(),
      transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
  ])

    try:
        img = Image.open(image_path).convert("RGB")
    except Exception as e:
        return {"error": f"Cannot open image: {e}", "reliable": False}

    try:
        img_tensor = transform(img).unsqueeze(0).to(device)
    except Exception as e:
        return {"error": f"Transform failed: {e}", "reliable": False}
    try:
        model.eval()
        with torch.no_grad():
            outputs    = model(img_tensor)
            probs      = torch.softmax(outputs, dim=1)
            confidence, predicted = probs.max(dim=1)
            confidence = confidence.item()
            predicted  = predicted.item()
    except Exception as e:
        return {"error": f"Prediction failed: {e}", "reliable": False}

    if confidence < threshold:
        return {
            "disease": "Uncertain — please try a clearer photo",,
            "confidence": round(confidence, 4),
            "reliable":   False
        }

    return {
        "disease":    label_to_name[predicted],
        "confidence": round(confidence, 4),
        "reliable":   True
    }

In [ ]:
print(torch.cuda.is_available())